## **1. Conectar drive e baixar o repositório**

In [1]:
import os
from google.colab import drive

if not os.path.exists("/content"):
    os.makedirs("/content", exist_ok=True)
%cd /content

drive.mount("/content/drive")

!rm -rf /content/segmentacao-lca
!git clone https://github.com/infocbra/segmentacao-lca.git /content/segmentacao-lca
%cd /content/segmentacao-lca

!pip install -q albumentations segmentation-models-pytorch

/content
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Cloning into '/content/segmentacao-lca'...
remote: Enumerating objects: 102, done.
remote: Counting objects: 100% (102/102), done.
remote: Compressing objects: 100% (92/92), done.
remote: Total 102 (delta 51), reused 26 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (102/102), 32.03 KiB | 537.00 KiB/s, done.
Resolving deltas: 100% (51/51), done.
/content/segmentacao-lca


## **2. Configurações**

In [6]:
import os
import time
import torch
import pandas as pd

from src.dados import criar_dataloaders
from src.modelos.unet import UNet
from src.modelos.unet_pretrained import criar_unet_pretrained
from src.treinamento import treinar_modelo
from src.avaliacao import avaliar_modelo
from src.predicao import gerar_predicoes_visuais

DISPOSITIVO = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo em uso:", DISPOSITIVO)

PASTA_BASE = "/content/drive/MyDrive/Segmentação_LCA/04_Dataset_Processado"
CAMINHO_SPLITS = "metadata/splits.csv"

# Escolha o cenario:
# Sequencia 1: "unet_zero"       | False | "unet_zero_orig"
# Sequencia 2: "unet_zero"       | True  | "unet_zero_aug"
# Sequencia 3: "unet_pretrained" | False | "unet_pre_orig"
# Sequencia 4: "unet_pretrained" | True  | "unet_pre_aug"
TIPO_MODELO = "unet_pretrained"
USAR_AUGMENTATION = True
CENARIO = "unet_pre_aug"

EPOCAS = 30
BATCH_SIZE = 8
TAXA_APRENDIZADO = 1e-4

PASTA_MODELOS = f"/content/drive/MyDrive/Segmentação_LCA/Modelos_salvos/{CENARIO}"
PASTA_RESULTADOS = f"/content/drive/MyDrive/Segmentação_LCA/Resultados/{CENARIO}"
os.makedirs(PASTA_MODELOS, exist_ok=True)
os.makedirs(PASTA_RESULTADOS, exist_ok=True)

def instanciar_modelo(tipo):
    if tipo == "unet_zero":
        return UNet(canais_entrada=1, canais_saida=1)
    elif tipo == "unet_pretrained":
        return criar_unet_pretrained()
    raise ValueError(f"Modelo desconhecido: {tipo}")

print("Configuracoes carregadas! Cenario:", CENARIO)

Dispositivo em uso: cuda
Configuracoes carregadas! Cenario: unet_pre_aug


## **3. Treinar e avaliar os 5 folds**

In [7]:
resultados_folds = []

for fold in range(1, 6):
    print(f"\n========== {CENARIO} | FOLD {fold} ==========")

    loader_treino, loader_val, loader_teste = criar_dataloaders(
        pasta_base=PASTA_BASE,
        caminho_splits=CAMINHO_SPLITS,
        fold_val=fold,
        batch_size=BATCH_SIZE,
        usar_augmentation=USAR_AUGMENTATION,
        num_workers=0,
    )

    modelo = instanciar_modelo(TIPO_MODELO)
    caminho_pesos = os.path.join(PASTA_MODELOS, f"fold_{fold}.pth")

    inicio = time.time()
    modelo, historico = treinar_modelo(
        modelo=modelo,
        loader_treino=loader_treino,
        loader_val=loader_val,
        dispositivo=DISPOSITIVO,
        epocas=EPOCAS,
        taxa_aprendizado=TAXA_APRENDIZADO,
        caminho_modelo=caminho_pesos,
    )
    tempo_segundos = time.time() - inicio

    # Salva historico de treino/validacao do fold
    pd.DataFrame(historico).to_csv(
        os.path.join(PASTA_RESULTADOS, f"historico_fold_{fold}.csv"),
        index=False,
    )

    # Avalia o melhor peso do fold no conjunto de teste fixo
    metricas_teste = avaliar_modelo(modelo, loader_teste, DISPOSITIVO)
    print(
        f"Teste Fold {fold} -> Dice: {metricas_teste['dice']:.4f} | "
        f"IoU: {metricas_teste['iou']:.4f} | Tempo: {tempo_segundos:.2f} segundos"
    )

    resultados_folds.append({
        "fold": fold,
        "dice_teste": metricas_teste["dice"],
        "iou_teste": metricas_teste["iou"],
        "tempo_seg": round(tempo_segundos, 2),
    })


========== unet_pre_aug | FOLD 1 ==========


/usr/local/lib/python3.13/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


Época 1/30 | Perda: 1.5448 | Dice: 0.0631 | IoU: 0.0326
Melhor modelo salvo!
Época 2/30 | Perda: 1.5168 | Dice: 0.0552 | IoU: 0.0284
Época 3/30 | Perda: 1.4945 | Dice: 0.0569 | IoU: 0.0293
Época 4/30 | Perda: 1.4758 | Dice: 0.0644 | IoU: 0.0334
Melhor modelo salvo!
Época 5/30 | Perda: 1.4611 | Dice: 0.0757 | IoU: 0.0396
Melhor modelo salvo!
Época 6/30 | Perda: 1.4377 | Dice: 0.0868 | IoU: 0.0456
Melhor modelo salvo!
Época 7/30 | Perda: 1.4260 | Dice: 0.1029 | IoU: 0.0545
Melhor modelo salvo!
Época 8/30 | Perda: 1.4033 | Dice: 0.1096 | IoU: 0.0582
Melhor modelo salvo!
Época 9/30 | Perda: 1.4055 | Dice: 0.0972 | IoU: 0.0519
Época 10/30 | Perda: 1.3749 | Dice: 0.1020 | IoU: 0.0544
Época 11/30 | Perda: 1.3684 | Dice: 0.0877 | IoU: 0.0464
Época 12/30 | Perda: 1.3513 | Dice: 0.0864 | IoU: 0.0456
Época 13/30 | Perda: 1.3423 | Dice: 0.0696 | IoU: 0.0363
Época 14/30 | Perda: 1.3228 | Dice: 0.0758 | IoU: 0.0395
Época 15/30 | Perda: 1.3077 | Dice: 0.1421 | IoU: 0.0786
Melhor modelo salvo!
Época 1

/usr/local/lib/python3.13/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


Época 1/30 | Perda: 1.8042 | Dice: 0.0682 | IoU: 0.0354
Melhor modelo salvo!
Época 2/30 | Perda: 1.7474 | Dice: 0.1079 | IoU: 0.0574
Melhor modelo salvo!
Época 3/30 | Perda: 1.7098 | Dice: 0.1433 | IoU: 0.0776
Melhor modelo salvo!
Época 4/30 | Perda: 1.6711 | Dice: 0.1680 | IoU: 0.0922
Melhor modelo salvo!
Época 5/30 | Perda: 1.6356 | Dice: 0.1807 | IoU: 0.1003
Melhor modelo salvo!
Época 6/30 | Perda: 1.6049 | Dice: 0.1805 | IoU: 0.1000
Época 7/30 | Perda: 1.5697 | Dice: 0.1953 | IoU: 0.1094
Melhor modelo salvo!
Época 8/30 | Perda: 1.5425 | Dice: 0.1973 | IoU: 0.1099
Melhor modelo salvo!
Época 9/30 | Perda: 1.5354 | Dice: 0.1705 | IoU: 0.0936
Época 10/30 | Perda: 1.4916 | Dice: 0.1656 | IoU: 0.0907
Época 11/30 | Perda: 1.4704 | Dice: 0.1401 | IoU: 0.0759
Época 12/30 | Perda: 1.4565 | Dice: 0.1641 | IoU: 0.0912
Época 13/30 | Perda: 1.4287 | Dice: 0.2311 | IoU: 0.1337
Melhor modelo salvo!
Época 14/30 | Perda: 1.4240 | Dice: 0.2567 | IoU: 0.1499
Melhor modelo salvo!
Época 15/30 | Perda: 1

/usr/local/lib/python3.13/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


Época 1/30 | Perda: 1.4563 | Dice: 0.0476 | IoU: 0.0244
Melhor modelo salvo!
Época 2/30 | Perda: 1.4130 | Dice: 0.0711 | IoU: 0.0370
Melhor modelo salvo!
Época 3/30 | Perda: 1.3948 | Dice: 0.1144 | IoU: 0.0607
Melhor modelo salvo!
Época 4/30 | Perda: 1.3720 | Dice: 0.1442 | IoU: 0.0780
Melhor modelo salvo!
Época 5/30 | Perda: 1.3438 | Dice: 0.1588 | IoU: 0.0865
Melhor modelo salvo!
Época 6/30 | Perda: 1.3415 | Dice: 0.1875 | IoU: 0.1039
Melhor modelo salvo!
Época 7/30 | Perda: 1.3240 | Dice: 0.1669 | IoU: 0.0939
Época 8/30 | Perda: 1.3291 | Dice: 0.2011 | IoU: 0.1160
Melhor modelo salvo!
Época 9/30 | Perda: 1.3081 | Dice: 0.2262 | IoU: 0.1328
Melhor modelo salvo!
Época 10/30 | Perda: 1.3051 | Dice: 0.2251 | IoU: 0.1357
Época 11/30 | Perda: 1.2815 | Dice: 0.1924 | IoU: 0.1168
Época 12/30 | Perda: 1.2831 | Dice: 0.1934 | IoU: 0.1189
Época 13/30 | Perda: 1.2673 | Dice: 0.1811 | IoU: 0.1084
Época 14/30 | Perda: 1.2534 | Dice: 0.1577 | IoU: 0.0915
Época 15/30 | Perda: 1.2388 | Dice: 0.1489 

/usr/local/lib/python3.13/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


Época 1/30 | Perda: 1.8034 | Dice: 0.0306 | IoU: 0.0156
Melhor modelo salvo!
Época 2/30 | Perda: 1.7754 | Dice: 0.0441 | IoU: 0.0226
Melhor modelo salvo!
Época 3/30 | Perda: 1.7375 | Dice: 0.0440 | IoU: 0.0225
Época 4/30 | Perda: 1.7006 | Dice: 0.0529 | IoU: 0.0272
Melhor modelo salvo!
Época 5/30 | Perda: 1.6963 | Dice: 0.0575 | IoU: 0.0297
Melhor modelo salvo!
Época 6/30 | Perda: 1.6384 | Dice: 0.0617 | IoU: 0.0319
Melhor modelo salvo!
Época 7/30 | Perda: 1.5954 | Dice: 0.0699 | IoU: 0.0364
Melhor modelo salvo!
Época 8/30 | Perda: 1.5745 | Dice: 0.0688 | IoU: 0.0358
Época 9/30 | Perda: 1.5576 | Dice: 0.0761 | IoU: 0.0399
Melhor modelo salvo!
Época 10/30 | Perda: 1.5471 | Dice: 0.0660 | IoU: 0.0347
Época 11/30 | Perda: 1.4907 | Dice: 0.0593 | IoU: 0.0310
Época 12/30 | Perda: 1.4709 | Dice: 0.0709 | IoU: 0.0376
Época 13/30 | Perda: 1.4399 | Dice: 0.0578 | IoU: 0.0301
Época 14/30 | Perda: 1.4378 | Dice: 0.0767 | IoU: 0.0405
Melhor modelo salvo!
Época 15/30 | Perda: 1.3918 | Dice: 0.0923 

/usr/local/lib/python3.13/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


Época 1/30 | Perda: 1.3850 | Dice: 0.0111 | IoU: 0.0056
Melhor modelo salvo!
Época 2/30 | Perda: 1.3711 | Dice: 0.0197 | IoU: 0.0101
Melhor modelo salvo!
Época 3/30 | Perda: 1.3552 | Dice: 0.0283 | IoU: 0.0148
Melhor modelo salvo!
Época 4/30 | Perda: 1.3360 | Dice: 0.0196 | IoU: 0.0100
Época 5/30 | Perda: 1.3261 | Dice: 0.0180 | IoU: 0.0091
Época 6/30 | Perda: 1.2958 | Dice: 0.0210 | IoU: 0.0107
Época 7/30 | Perda: 1.2940 | Dice: 0.0364 | IoU: 0.0190
Melhor modelo salvo!
Época 8/30 | Perda: 1.2492 | Dice: 0.0498 | IoU: 0.0271
Melhor modelo salvo!
Época 9/30 | Perda: 1.2458 | Dice: 0.0549 | IoU: 0.0303
Melhor modelo salvo!
Época 10/30 | Perda: 1.2208 | Dice: 0.0242 | IoU: 0.0124
Época 11/30 | Perda: 1.1955 | Dice: 0.0635 | IoU: 0.0348
Melhor modelo salvo!
Época 12/30 | Perda: 1.1789 | Dice: 0.0874 | IoU: 0.0510
Melhor modelo salvo!
Época 13/30 | Perda: 1.1446 | Dice: 0.0741 | IoU: 0.0433
Época 14/30 | Perda: 1.1485 | Dice: 0.1289 | IoU: 0.0868
Melhor modelo salvo!
Época 15/30 | Perda: 1

## **4. Resumo final dos 5 folds**

In [8]:
df_res = pd.DataFrame(resultados_folds)
df_res.to_csv(os.path.join(PASTA_RESULTADOS, "resumo_teste.csv"), index=False)

print(df_res)
print(f"\nDice medio: {df_res['dice_teste'].mean():.4f} +/- {df_res['dice_teste'].std():.4f}")
print(f"IoU medio:  {df_res['iou_teste'].mean():.4f} +/- {df_res['iou_teste'].std():.4f}")

   fold  dice_teste  iou_teste  tempo_seg
0     1    0.670850   0.513776      15.69
1     2    0.648101   0.494305      17.89
2     3    0.681016   0.522975      14.13
3     4    0.512844   0.359499      17.01
4     5    0.549245   0.405663      13.46

Dice medio: 0.6124 +/- 0.0763
IoU medio:  0.4592 +/- 0.0726


## **5. Predições no Conjunto de Teste**

In [9]:
melhor_fold = int(df_res.loc[df_res["dice_teste"].idxmax(), "fold"])
print(f"Gerando figuras visuais com o melhor fold (Fold {melhor_fold})...")

_, _, loader_teste = criar_dataloaders(
    pasta_base=PASTA_BASE,
    caminho_splits=CAMINHO_SPLITS,
    fold_val=melhor_fold,
    batch_size=BATCH_SIZE,
    usar_augmentation=False,
    num_workers=0,
)

gerar_predicoes_visuais(
    modelo=instanciar_modelo(TIPO_MODELO),
    caminho_pesos=os.path.join(PASTA_MODELOS, f"fold_{melhor_fold}.pth"),
    loader_teste=loader_teste,
    pasta_saida=os.path.join(PASTA_RESULTADOS, "figuras_teste"),
    dispositivo=DISPOSITIVO,
)

Gerando figuras visuais com o melhor fold (Fold 3)...
4 figuras salvas em: /content/drive/MyDrive/Segmentação_LCA/Resultados/unet_pre_aug/figuras_teste
